In [1]:
%pip install torch torchvision transformers pillow tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import os
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

c:\Users\arthu\miniconda3\envs\sandbox\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 426.74it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may

In [8]:
prompts = [
    "a photo",
    "a drawing",
]

In [9]:
refined_prompts = [
    "a single chair",
    "more than one chair",
]

In [10]:
def score_image(image_path, prompts=prompts):
    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        text=prompts,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)

    return probs.cpu().numpy()[0]

In [11]:
import shutil
from pathlib import Path

input_folder = Path("raw_images")
output_folder = Path("filtered_chairs")
output_folder.mkdir(parents=True, exist_ok=True)

valid_ext = {".jpg", ".jpeg", ".png"}

image_paths = [
    p for p in input_folder.rglob("*")
    if p.is_file() and p.suffix.lower() in valid_ext
]

kept = 0
errors = 0

for path in tqdm(image_paths):
    try:
        probs = score_image(str(path), prompts=prompts)
        chair_white_score = probs[0]  # premier prompt


        if chair_white_score > 0.7:
            refined_probs = score_image(str(path), prompts=["a chair on a white background", "a chair on a black background"])
            if refined_probs[0] > 0.7:
                rel_path = path.relative_to(input_folder)
                dst_path = output_folder / rel_path
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, dst_path)
                kept += 1
    except Exception:
        errors += 1
        continue

print(f"Images analysées: {len(image_paths)} | conservées: {kept} | erreurs: {errors}")

100%|██████████| 1694/1694 [06:21<00:00,  4.44it/s]

Images analysées: 1694 | conservées: 383 | erreurs: 0
